# 00 · End-to-end M5 pipeline

Drives `m5.cv` + `m5.evaluation` against the prepared long-frame parquet.
Every cell here is a thin call into the package — equivalent to 
`make cv-stats && make cv-lgbm` plus inline scoring.

**Prereqs:** `make bootstrap && make prep` so `data/processed/long.parquet` exists.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import lgbm_cv, stats_cv
from m5.evaluation import compute_components, wrmsse_for_models
from m5.plots import configure_style

configure_style()
set_global_seed()

df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')
df.head()

,unique_id,item_id,dept_id,cat_id,store_id,state_id,d,y,ds,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1741,0.0,2015-11-04,11540,none,none,none,none,1,0,0,2.24
1,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1742,0.0,2015-11-05,11540,none,none,none,none,1,1,1,2.24
2,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1743,0.0,2015-11-06,11540,none,none,none,none,1,1,1,2.24
3,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1744,1.0,2015-11-07,11541,none,none,none,none,1,1,0,2.24
4,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1745,0.0,2015-11-08,11541,none,none,none,none,1,0,1,2.24


## Statistical baselines (Theta + AutoETS + SeasonalNaive)

In [3]:
cv_stats = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
components = compute_components(df[df['ds'] < cv_stats['ds'].min()])
stats_scores = wrmsse_for_models(cv_stats[['unique_id', 'ds', 'y']], cv_stats, components)
stats_scores

12:49:05 | INFO    | m5.cv:stats_cv:46 - stats_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/Git

AutoETS          0.825115
Theta            0.833672
SeasonalNaive    1.048186
Name: wrmsse, dtype: float64

## LightGBM global model

In [4]:
cv_lgbm = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
lgbm_scores = wrmsse_for_models(cv_lgbm[['unique_id', 'ds', 'y']], cv_lgbm, components)
lgbm_scores

12:49:13 | INFO    | m5.cv:lgbm_cv:68 - lgbm_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:683: UserWarning: The following series were dropped completely due to the transformations and features: ['FOODS_3_595_CA_1', 'HOUSEHOLD_1_020_WI_2', 'HOUSEHOLD_1_278_CA_3'].
These series won't show up if you use `MLForecast.forecast_fitted_values()`.
You can set `dropna=False` or use transformations that require less samples to mitigate this
  warnings.warn(
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/home/ric

LGBM    0.818543
Name: wrmsse, dtype: float64

## Combined leaderboard

Lower is better. For the official 12-level WRMSSE breakdown, run `06_score.ipynb`.

In [5]:
leaderboard = pd.concat([stats_scores, lgbm_scores]).sort_values()
leaderboard.to_frame('WRMSSE')

,WRMSSE
LGBM,0.818543
AutoETS,0.825115
Theta,0.833672
SeasonalNaive,1.048186
